# Wundernn GRU — resumable GPU training (Colab)

Runs `02_trunk_global_full`'s recipe (trunk 256-128 + ridge-init skip + GRU 128x2 + global WP loss) at a large
batch on a T4. **Run top to bottom.** Edit only the two constants marked `EDIT` in the first cell.

* **First session: leave `SMOKE = True`.** It runs the whole pipeline for ~20 steps, kills itself on purpose at
  step 10, resumes, finishes, calibrates and writes `summary.md` (a few minutes of units). Only when it prints
  `SMOKE OK`, set `SMOKE = False` and run again for the real run.
* **After a disconnect / runtime reset: just re-run the notebook.** Training uses `--resume auto`; checkpoints are
  mirrored to Google Drive every 200 steps, so even a wiped `/content` continues where it left off.
* Needs Colab secret `GITHUB_TOKEN` (key icon in the left bar, "Notebook access" ON): a fine-grained token with
  *Contents: read & write* on the repo only. The token is never printed or written to disk.
* Drive folder must contain `train_subset.parquet` and `valid_subset.parquet`.

In [ ]:
# ============================ EDIT THESE TWO ============================
REPO_URL  = "https://github.com/DmitryTsybulkin/wundernn-competition.git"   # private repo, https form
DRIVE_DIR = "/content/drive/MyDrive/wnn"      # contains train_subset.parquet + valid_subset.parquet
# ========================================================================

# ---- defaults: leave alone unless you have a reason -----------------------------------------
SMOKE      = True          # True: ~20-step end-to-end check (kill + resume included). False: the real run.
RUN_NAME   = "05_gru_gpu"  # runs/<RUN_NAME>; next free NN in the repo's runs/
BATCH_SIZE = 256           # 128 / 256 / 512 are the sizes worth trying (notes/LOG.md 2026-09-21)
LR         = None          # None = sqrt scaling from 3e-4 @ batch 64  (6e-4 @ 256, 8.5e-4 @ 512)
MAX_HOURS  = 3.0           # training-time budget, summed across resumes
DEVICE     = "cuda"        # 'cpu' only for a local rehearsal of this notebook
EXTRA      = []            # advanced: extra train.train flags appended last, e.g. ["--lr-schedule", "cosine"]
GITHUB_SECRET = "GITHUB_TOKEN"
T4_UNITS_PER_HOUR = 1.9    # ESTIMATE of Colab's T4 burn rate; the Colab resources panel is authoritative
AUTO_RELEASE_RUNTIME = False   # True: release the runtime after a successful push (stops billing)
WORK_DIR = "/content"

import sys, time
T_NOTEBOOK_START = time.time()
print(f"SMOKE={SMOKE} RUN_NAME={RUN_NAME} BATCH_SIZE={BATCH_SIZE} LR={LR or 'sqrt-scaled'} MAX_HOURS={MAX_HOURS}")

## 1. GPU

In [ ]:
import subprocess
if DEVICE == "cuda":
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    import torch
    if r.returncode != 0 or not torch.cuda.is_available():
        raise RuntimeError("NO GPU ASSIGNED. Runtime > Change runtime type > T4 GPU, then Runtime > Restart session and run all.")
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print(f"DEVICE={DEVICE}: GPU check skipped (local rehearsal)")

## 2. Google Drive + data (copied to local disk once; re-runs skip the copy)

In [ ]:
import os, shutil
from pathlib import Path
if DEVICE == "cuda":
    from google.colab import drive
    drive.mount("/content/drive")
DRIVE = Path(DRIVE_DIR)
src = {"train": DRIVE / "train_subset.parquet", "valid": DRIVE / "valid_subset.parquet"}
missing = [str(p) for p in src.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"missing on Drive: {missing}. Upload datasets/train_subset.parquet and datasets/valid_subset.parquet to {DRIVE}.")
LOCAL_DATA = Path(WORK_DIR) / "data"
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
DATA = {}
for k, p in src.items():
    dst = LOCAL_DATA / p.name
    if not dst.exists() or dst.stat().st_size != p.stat().st_size:
        t = time.time()
        print(f"copying {p.name} ({p.stat().st_size/1e9:.2f} GB) Drive -> local disk ...", flush=True)
        shutil.copyfile(p, str(dst) + ".part")
        os.replace(str(dst) + ".part", dst)
        print(f"  done in {time.time()-t:.0f}s")
    DATA[k] = dst
    print(f"{k}: {dst} ({dst.stat().st_size/1e9:.2f} GB)")

## 3. Clone the private repo (token from Colab secrets — never printed, never stored in the remote URL)

In [ ]:
REPO = Path(WORK_DIR) / "wnn_repo"
if DEVICE == "cuda":
    from google.colab import userdata
    try:
        _token = userdata.get(GITHUB_SECRET)
    except Exception as e:
        raise RuntimeError(f"Colab secret {GITHUB_SECRET!r} is missing or notebook access is off (left bar > key icon).") from None
else:
    _token = os.environ.get("GITHUB_TOKEN_REHEARSAL", "unused")
ASKPASS = Path(WORK_DIR) / ".git_askpass.sh"
ASKPASS.write_text('#!/bin/sh\ncase "$1" in Username*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
ASKPASS.chmod(0o700)
GIT_ENV = dict(os.environ, GIT_ASKPASS=str(ASKPASS), GIT_TOKEN=_token, GIT_TERMINAL_PROMPT="0")

def git(*args, cwd=REPO, check=True):
    r = subprocess.run(["git", *args], cwd=cwd, env=GIT_ENV, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_token, "***")
    if check and r.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {r.returncode}):\n{out}")
    return out.strip()

if (REPO / ".git").exists():
    print(git("fetch", "origin"))
    try:
        print(git("merge", "--ff-only", "@{u}"))
    except RuntimeError as e:
        print("WARNING: could not fast-forward the existing clone; using it as is.\n", e)
else:
    print(git("clone", REPO_URL, str(REPO), cwd=WORK_DIR))
git("config", "user.name", "colab-runner")
git("config", "user.email", "colab-runner@users.noreply.github.com")
print("code under test:", git("log", "-1", "--format=%h %ad %s", "--date=short"))
for needed in ("train/train.py", "colab/launch.py", "models/ridge_v1.npz", "utils.py"):
    if not (REPO / needed).exists():
        raise FileNotFoundError(f"{needed} is not in the cloned repo — commit and push it first (the clone only sees committed files).")

## 4. Dependencies (only what Colab lacks; the training code needs just torch, numpy, pyarrow)

In [ ]:
import importlib.util
missing = [m for m in ("numpy", "pyarrow", "torch") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy, pyarrow, torch
print(f"numpy {numpy.__version__}  pyarrow {pyarrow.__version__}  torch {torch.__version__}  (wandb not needed: --wandb-mode disabled)")

## 5. Train (`--resume auto`)
`SMOKE=True` → kill-at-step-10 + resume + finish + summary, then prints `SMOKE OK`. `SMOKE=False` → the real run; **re-run
this cell after any disconnect** and it continues. The exact command is printed first.

In [ ]:
DRIVE_RUN = Path(DRIVE_DIR) / "runs" / RUN_NAME
cmd = [sys.executable, "colab/launch.py", "--run-name", RUN_NAME,
       "--train-path", str(DATA["train"]), "--valid-path", str(DATA["valid"]),
       "--out-dir", str(REPO / "runs" / RUN_NAME), "--mirror-dir", str(DRIVE_RUN),
       "--device", DEVICE, "--batch-size", str(BATCH_SIZE), "--max-hours", str(MAX_HOURS)]
if LR is not None:
    cmd += ["--lr", str(LR)]
if SMOKE:
    cmd.append("--smoke")
if EXTRA:
    cmd += ["--extra", *EXTRA]
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in p.stdout:
        print(line, end="", flush=True)
except KeyboardInterrupt:            # "Interrupt execution": let train.py checkpoint, then exit
    p.terminate()
    for line in p.stdout:
        print(line, end="", flush=True)
rc = p.wait()
print(f"\nlauncher exit code: {rc}", "(interrupted or crashed: re-run this cell to resume)" if rc else "")

## 6. Commit and push the run back to the repo (skipped in smoke mode)

In [ ]:
if SMOKE:
    print("SMOKE=True: nothing to push. If the log above says SMOKE OK, set SMOKE=False and run again.")
else:
    run_dir = REPO / "runs" / RUN_NAME
    names = ["summary.md", "log.txt", "config.json", "best.pt", "last.pt", "ckpt_latest.pt"]
    present = [n for n in names if (run_dir / n).exists()]
    if not present:
        print("nothing to push yet.")
    else:
        git("add", "-f", *[f"runs/{RUN_NAME}/{n}" for n in present])
        if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO).returncode == 0:
            print("no changes to commit.")
        else:
            print(git("commit", "-m", f"runs/{RUN_NAME}: results from Colab ({', '.join(present)})"))
        try:
            git("pull", "--rebase", "origin", "HEAD")
        except RuntimeError as e:
            print("WARNING: pull --rebase failed, pushing anyway:", e)
        print(git("push", "origin", "HEAD"))
        PUSHED = True

## 7. Elapsed time and compute-unit estimate

In [ ]:
train_s = None
ck = REPO / "runs" / RUN_NAME / "ckpt_latest.pt"
if ck.exists():
    c = torch.load(ck, map_location="cpu", weights_only=False)
    train_s = c["elapsed"]
    print(f"run {RUN_NAME}: step {c['global_step']}, finished={c['finished']}, best in-run wp_masked={c['best']['wp']:.4f} @ step {c['best']['step']}")
    print(f"training time, summed over {len(c.get('resume_log', [])) + 1} launch(es): {train_s/3600:.2f} h")
session_h = (time.time() - T_NOTEBOOK_START) / 3600
print(f"this notebook session so far: {session_h:.2f} h  ->  ~{session_h * T4_UNITS_PER_HOUR:.1f} compute units "
      f"(assumes {T4_UNITS_PER_HOUR} units/h for a T4; the runtime is billed until released, so check the Colab resources panel)")
if AUTO_RELEASE_RUNTIME and not SMOKE and globals().get("PUSHED"):
    from google.colab import runtime
    runtime.unassign()